# Why the EDM UNet Doesn't Memorize from Pure Noise — Mechanism

`edm_unet_train_size_sweep.ipynb` found that the EDM UNet produces novel samples at every
n_train ∈ [4, 32]; `edm_unet_train_size_sweep_sigma_fix.ipynb` ruled out the sampler
(σ-range and discretization ablations changed nothing). This notebook assembles three
diagnostics that together identify the actual mechanism.

**The three diagnostics** (heavy computation was run by scripts; this notebook loads the
saved artifacts from `results/data/` and reproduces the analysis/figures):

1. **Train/held-out denoiser gap** (`edm_unet_denoiser_gap.pt`) — does the trained denoiser
   know anything sample-specific, and at which noise levels?
2. **Basin reachability** (`edm_unet_basin_reconstruction.pt`) — start the reverse SDE from
   noised *training* images y + σ₀ε: do memorized basins exist, and from how far away are
   they attractive?
3. **CoordConv ablation** (`edm_unet_coordconv.pt`) — give the UNet coordinate channels
   (breaking translation equivariance, the standard suspect) and retrain: does memorized
   generation appear?

**Mechanism found:**

- The UNet **does memorize** — its denoising error on training images is up to ~15× lower
  than on held-out fields (n_train=4), but only at **small σ ≲ 2**. At the large σ where the
  reverse SDE decides which basin to enter, the gap is 1.0: no sample-specific knowledge.
  The GMM calibrator has an astronomical gap at *every* σ — in d = 128² dimensions the
  posterior concentrates on the correct training image even at σ = 10, so a full memorizer
  is sample-specific at all noise levels. The UNet only is near the data.
- The memorized basins are **real and strongly attractive**: from y + σ₀ε the sampler
  returns to its exact source image with 100% identity accuracy at every σ₀ tested (up to
  10), with reconstruction error growing as each frequency band of the source drowns
  (per-mode amplitude: coarse ≈ 18.5, mid1 ≈ 1.6, mid2 ≈ 0.40, fine ≈ 0.17 — the source's
  coarse imprint survives even σ₀ = 10, which is what preserves identity).
- **CoordConv does not help** (coarse ratio ≈ baseline). Position information is not the
  missing piece — local memorization already works without it.

**One-line summary: the UNet is an *amplifier* of the coarse content present in its
initialization; the GMM is a *classifier* of it.** Given a noised training image, the
surviving coarse imprint is amplified back into that image (reconstruction works, identity
kept). Given pure noise, the random coarse pattern is amplified into a *novel* field — the
UNet lacks the global template-matching map at large σ that snaps arbitrary noise to the
nearest training image, which the GMM has by construction. Memorization is present in the
network but unreachable from pure noise.

**Why the UNet can't learn the large-σ snap:** identifying which training image underlies
heavily-noised input requires whole-image matched filtering against stored templates. A
small, local, translation-equivariant conv net can overfit local structure near the data
(small σ) but has no mechanism for global template identification — and the EDM σ-sampling
(LogNormal(−1.2, 1.2)) barely trains σ > 5 anyway.

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
from multiband_data_utils import make_radial_k_grid

data_dir = os.path.join(repo_root, 'results', 'data')
gap   = torch.load(os.path.join(data_dir, 'edm_unet_denoiser_gap.pt'), weights_only=False)
basin = torch.load(os.path.join(data_dir, 'edm_unet_basin_reconstruction.pt'), weights_only=False)
coord = torch.load(os.path.join(data_dir, 'edm_unet_coordconv.pt'), weights_only=False)
ablat = torch.load(os.path.join(data_dir, 'edm_unet_sigma_fix_ablation.pt'), weights_only=False)
covtk = torch.load(os.path.join(data_dir, 'gmm_covariance_tikhonov_sweep.pt'), weights_only=False)

bands = ablat['bands']
k_centers = ablat['k_centers']
print('artifacts loaded:', [k for k in ['gap', 'basin', 'coord', 'ablat', 'covtk']])

## Diagnostic 1 — Train/held-out denoiser gap

Denoising MSE of the saved checkpoints on training vs held-out images, per σ. A gap > 1
means the denoiser is sample-specific at that noise level. The GMM posterior-mean denoiser
(perfect memorizer) has train error ≈ 0 at every σ — its gap is ~10¹² — shown as text, not
plotted.

In [ ]:
sigmas = gap['sigmas']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(gap['results'])))
for color, (n_train, rows) in zip(colors, sorted(gap['results'].items())):
    g = [r['unet_held'] / max(r['unet_train'], 1e-12) for r in rows]
    axes[0].plot(sigmas, g, 'o-', color=color, label=f'n_train={n_train}')
axes[0].axhline(1.0, ls='--', color='black', lw=1, label='no gap (nothing sample-specific)')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('noise level σ'); axes[0].set_ylabel('held-out MSE / train MSE')
axes[0].set_title('UNet denoiser gap: memorization exists, but only at small σ')
axes[0].legend(fontsize=9)

rows4 = gap['results'][4]
axes[1].loglog(sigmas, [r['unet_train'] for r in rows4], 'o-', color='tab:blue', label='train images')
axes[1].loglog(sigmas, [r['unet_held'] for r in rows4], 's-', color='tab:red', label='held-out images')
axes[1].set_xlabel('noise level σ'); axes[1].set_ylabel('denoising MSE')
axes[1].set_title('n_train = 4: absolute errors')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

print('GMM calibrator: train MSE ~ 0 (float precision) at EVERY σ -> gap ~ 1e12.')
print('A full memorizer is sample-specific at all noise levels; the UNet only near the data.')

## Diagnostic 2 — Basin reachability

Reverse SDE started from y_i + σ₀ε (n_train = 4 checkpoint). `rel_src` = relative L2 of the
result to its source image, `rel_other` = to the best *other* training image. Identity
accuracy was 100% at every σ₀ (up to 10) — the memorized basins are strongly attractive.
The error growth with σ₀ is the multiscale part of the story: each frequency band of the
source is lost once σ₀ exceeds its per-mode amplitude √λ(k), fine first, coarse never
(√λ_coarse ≈ 18.5 > 10).

In [ ]:
rows = basin['rows']
s0 = [r['sigma0'] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
axes[0].plot(s0, [r['unet_rel_src'] for r in rows], 'o-', color='tab:blue', lw=2,
             label='UNet: error to source image')
axes[0].plot(s0, [r['unet_rel_other'] for r in rows], 's--', color='tab:blue', alpha=0.5,
             label='UNet: error to best other image')
axes[0].plot(s0, [r['gmm_rel_src'] for r in rows], 'o-', color='tab:red', lw=1.5,
             label='GMM: error to source image')
# per-band amplitude markers: where each band of the source drowns
lam = covtk['lam']; kr = make_radial_k_grid(128)
for name, style in [('fine', ':'), ('mid2', ':'), ('mid1', ':')]:
    m = (kr >= bands[name][0]) & (kr < bands[name][1])
    amp = lam[m].mean().sqrt().item()
    axes[0].axvline(amp, ls=style, color='gray', lw=1)
    axes[0].text(amp, 1.32, f'{name}\ndrowns', ha='center', fontsize=7, color='gray')
axes[0].set_xscale('log')
axes[0].set_xlabel('start noise level σ₀'); axes[0].set_ylabel('relative L2 error')
axes[0].set_title('Reconstruction from noised training images (identity kept 100% at all σ₀)')
axes[0].legend(fontsize=9, loc='center left')

# image strip: source and reconstructions at increasing sigma0
show = [r for r in rows if r['sigma0'] in (0.5, 2.0, 5.0, 10.0)]
axes[1].axis('off')
strip = torch.cat([basin['x_train'][0]] + [r['unet_recs'][0] for r in show], dim=1)
axes[1].imshow(strip.numpy(), cmap='RdBu')
axes[1].set_title('source | recon @ σ₀=0.5 | 2 | 5 | 10   (coarse blobs persist, fine renewed)',
                  fontsize=10)
plt.tight_layout(); plt.show()

print(f"{'sigma0':>7} | {'rel_src':>8} {'rel_other':>10} {'id_acc':>7}")
for r in rows:
    print(f"{r['sigma0']:>7.1f} | {r['unet_rel_src']:>8.4f} {r['unet_rel_other']:>10.4f} "
          f"{r['unet_id_acc']:>7.2f}")

## Diagnostic 3 — CoordConv ablation

Same architecture + two coordinate input channels (absolute position), trained identically,
sampled with the fixed sampler (σ_max = 10, 1000 steps). If translation equivariance were
the bottleneck, memorized generation should appear here.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
kc = k_centers.numpy()
for color, n_train in zip(['tab:blue', 'tab:orange'], coord['n_train_values']):
    ax.plot(kc, coord['results'][n_train]['mean_ratio'].numpy(),
            color=color, lw=1.8, label=f'CoordConv n_train={n_train}')
    ax.plot(kc, ablat['sweep_results'][n_train]['unet']['D']['mean_ratio'].numpy(),
            color=color, lw=1.4, ls='--', alpha=0.7, label=f'baseline n_train={n_train}')
    ax.plot(kc, ablat['sweep_results'][n_train]['gmm']['D']['mean_ratio'].numpy(),
            color=color, lw=1.0, ls=':', alpha=0.6, label=f'GMM n_train={n_train}')
ax.axhline(1.0, ls=':', color='black', lw=1)
for bname, bcolor in [('coarse', 'tab:blue'), ('fine', 'tab:red')]:
    lo, hi = bands[bname]
    ax.axvspan(lo, hi, color=bcolor, alpha=0.08)
ax.set_yscale('log')
ax.set_xlabel('wavenumber k'); ax.set_ylabel('mean(err_to_NN / err_to_random)')
ax.set_title('CoordConv ≈ baseline: absolute position is not the missing ingredient')
ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

print(f"{'n_train':>8} | {'CoordConv coarse':>16} | {'baseline coarse':>15} | {'GMM coarse':>10}")
for n_train in coord['n_train_values']:
    cc = coord['results'][n_train]['coarse_score'].mean().item()
    bl = ablat['sweep_results'][n_train]['unet']['D']['coarse_score'].mean().item()
    gm = ablat['sweep_results'][n_train]['gmm']['D']['coarse_score'].mean().item()
    print(f'{n_train:>8d} | {cc:>16.4f} | {bl:>15.4f} | {gm:>10.4f}')

## Implications for the paper

1. **Memorization is not an automatic consequence of tiny datasets.** The same training
   objective at its stationary point (GMM) memorizes fully; a 195k-parameter conv UNet
   trained on the same 4 images memorizes only *locally* around the data and generates
   novel fields from pure noise. The architecture's (in)ability to perform global template
   identification at large σ — not data size, training budget, or sampler settings —
   determines whether memorized generation occurs.
2. **Amplifier vs classifier.** The reverse dynamics of the UNet amplify whatever coarse
   imprint the initialization carries (100% source identity from y + 10ε); the GMM
   classifies any initialization to a training template. This cleanly separates two roles
   usually conflated in memorization discussions: storing the data (the UNet does) and
   routing noise to it (the UNet doesn't).
3. **Multiscale structure of memory.** The reconstruction error vs σ₀ traces the per-band
   amplitudes √λ(k): fine content of a memorized image is lost first (σ ≈ 0.17), coarse
   last (σ ≈ 18.5). This is the inference-side mirror of the per-wavenumber memorization
   ratio used throughout the project, and connects directly to the covariance-weighted
   Tikhonov experiment (Σ's spectrum controls both).
4. **Consequence for the regularization thread:** UNet + Tikhonov comparisons should be
   read as regularizing a *partial, locally-memorizing* model; the GMM remains the right
   testbed for regularizer design (as in `gmm_covariance_tikhonov_memorization.ipynb`), and
   any UNet memorization study on this data needs either a global mechanism in the
   architecture (attention / matched filtering) or should evaluate memorization via
   reconstruction (basin) metrics rather than pure-noise generation.

**Follow-ups (cheap first):** band-resolved reconstruction error vs σ₀ (per-scale
"memorization horizon" figure — rerun `basin_reconstruction` keeping full reconstructions);
attention/bottleneck-global UNet variant as the constructive fix; reconstruction-based
memorization metric for the Tikhonov sweeps.